# Lab 9: Report

## Three-Point Bending Test of Glass

The main objective of this lab is to understand the testing methods and related challenges for determining the mechanical properties of brittle materials, such as glasses and ceramics.

Recommended reading: Callister, *Materials Science and Engineering: An Introduction*, Sections 12.8–12.9

## 1. Importing data

### 1.1 Definitions

Complete the code cell below and run it to:

- import the necessary libraries
- define the span used for the three-point bend test, in millimeters

In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# define the span used to collect the data in millimeters
SPAN_MM = YOUR RESPONSE # millimeters


### 1.2 Import the data

The data from each lab section has been compiled into a single `.csv` file for analysis. Import the `.csv` data file as a pandas dataframe.

The code below:

- skips the first row
- only imports the columns you need
- defines the column names during the import

Run the code and inspect the output.

In [ ]:
# import the data from the .csv file, ignoring the first column
df = pd.read_csv('GlassFractureDataCleaned.csv',
                 skiprows=1,
                 usecols=[1,2,4],
                 names=["Label", "Dia / mm", "Load / N"])

# convert all labels to lowercase so that the comparison is not case-sensitive
df["Label"] = df["Label"].str.lower()

df.head()

### 1.3 Look for outliers

The glass rods should all have a diameter of either ~4 or ~8 mm and break at distinctly different loads. Occasionally, a sample is mislabeled. Run the code below to sort the data by diameter and look for outliers.

In [ ]:
df_grouped = df.copy()
dia = df_grouped["Dia / mm"]

# Assign diameter categories (ordered, so sorting follows 4 mm → 8 mm → Other)
df_grouped["Diameter group"] = pd.Categorical(
    np.select([dia.between(3.5, 4.5), dia.between(7.5, 8.5)], ["4 mm", "8 mm"], default="Other"),
    categories=["4 mm", "8 mm", "Other"], ordered=True,
)
df_grouped = df_grouped.sort_values(["Diameter group", "Dia / mm"])

is_other = df_grouped["Diameter group"] == "Other"
main = df_grouped[~is_other]   # only 4 mm and 8 mm

# Load outliers within each diameter category (1.5×IQR rule)
loads = main.groupby("Diameter group", observed=True)["Load / N"]
q1 = loads.transform(lambda s: s.quantile(0.25))
q3 = loads.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
is_outlier = ~main["Load / N"].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

print("Load statistics by diameter:")
display(loads.agg(["count", "mean", "std"]).round(1))

print("Load outliers (1.5×IQR rule):")
display(main[is_outlier])

if not df_grouped.loc[is_other].empty:
    print("Specimens outside the 4 mm and 8 mm diameter categories:")
    display(df_grouped.loc[is_other])

**`ENGINEERING CHECK`**: Inspect the identified outliers above. Look for:

- abnormally high loads for the 4 mm specimens
- abnormally low loads for the 8 mm specimens
- unreasonable diameters

You can drop a specimen with the cell below. After you delete an entry, re-run the cell above.

In [ ]:
df = df.drop(index= YOUR RESPONSE) # replace YOUR RESPONSE with the index on the left side of the dataframe to delete a row

### 1.4 Check the labels

Every specimen must be labeled either `coated` or `uncoated` to be sorted into a group. Run the cell below to list any specimens with a different or missing label.

In [ ]:
# list the specimens whose label is not exactly "coated" or "uncoated"
bad_labels = df[~df["Label"].isin(["coated", "uncoated"])]
display(bad_labels)

**`ENGINEERING CHECK`**: For each specimen listed above, either

- re-label it, if the intended label is clear (for example, a typo in the `.csv` file), or
- delete it, if the coating cannot be identified.

Use the cells below, then re-run the cell above to confirm that no specimens are listed.

In [ ]:
df.loc[200, 'Label'] = "YOUR RESPONSE"  # Example of re-labeling a specimen: replace 200 with the index and "YOUR RESPONSE" with "coated" or "uncoated"

In [ ]:
df = df.drop(index= YOUR RESPONSE) # use the index on the left side of the dataframe to delete a row

## 2. Group the data and calculate stress

In this section you will prepare the data for plotting and statistical analysis.


### 2.1 Make groups

Each sample must be sorted into one of four groups based on diameter and coating:

| | uncoated | coated |
|:---|:---|:---|
|**4 mm** | group 1 | group 2 |
|**8 mm** | group 3 | group 4 |


The code below specifies the conditions a sample must meet to belong to each group. Each sample is assigned a group number **1** through **4**.

**Note**: If a sample can't be identified it will be assigned to group **0**. After the label check in Section 1.4, group 0 should be empty.

In [ ]:
conditions = [
    (df['Label'] == 'uncoated') & (df['Dia / mm'] >= 3) & (df['Dia / mm'] <= 5),
    (df['Label'] == 'coated') & (df['Dia / mm'] >= 3) & (df['Dia / mm'] <= 5),
    (df['Label'] == 'uncoated') & (df['Dia / mm'] >= 7) & (df['Dia / mm'] <= 9),
    (df['Label'] == 'coated') & (df['Dia / mm'] >= 7) & (df['Dia / mm'] <= 9)
]

df['group'] = np.select(condlist = conditions, choicelist = [1, 2, 3, 4], default=0)
for i in np.unique(df["group"]):
    group_count = (df["group"] == i).sum()
    print(f"Group {i}, {group_count} entries:")
    display(df[df['group'] == i].head(5))


**`ENGINEERING CHECK`**:

If any samples are still in group 0, either return to Section 1.4 to re-label or delete them and then re-run the cells from there, or run the cell below to delete all specimens in group 0.

In [ ]:
df = df[df['group'] != 0].copy()  # Delete all specimens in group 0


### 2.2 Calculations

All specimens should now be sorted into groups 1-4. 

Complete the equation below for the maximum tensile stress in the glass rod based on the load at fracture and the specimen geometry. Define the variables:

$$\sigma_\mathrm{fs}=...$$


**Variables**:

$\sigma_\mathrm{fs}$: flexural strength (maximum tensile stress at fracture) / MPa

In the code cell below, calculate the flexural strength in MPa. Round to appropriate significant figures.

**Functions for your code**:

- `.round(n)`: round to n decimal places
- `**n`: raise to the power of n

**Variables for your code**:

- `df['Load / N']`: Load at fracture
- `df['Dia / mm']`: Specimen diameter in millimeters
- `SPAN_MM`: Span length of test frame in millimeters

In [ ]:
df['Flex Strength / MPa'] = YOUR RESPONSE
df['Flex Strength / MPa'] = YOUR RESPONSE

display(df.head())   # display the first few lines of the dataframe after calculating flexural strength

print("Flex Strength Raw Statistics:")
display(df['Flex Strength / MPa'].agg(["count", "mean", "std"]).to_frame().T.round(1))

**`ENGINEERING CHECK`**:

Compare the calculated values of flexural strength with accepted values. Are you in the correct range? If not, revisit your calculation.

## 3. Generating Figures

### 3.1 Distribution of failed specimens as a function of flexural strength (20 points) 

Use a histogram to plot the number of failed samples as a function of flexural strength for all four groups.

1. Define `STEP`: the width in MPa of each bin
1. Define `STRESS_MIN` and `STRESS_MAX`: Stress cutoffs for the plot
1. Select which plotting option displays the data best (delete the other code)
1. Label the axes
1. Provide a descriptive figure caption
1. Explain your choice

Complete the code cells below and run them. If any specimens have been ignored, consider whether your stress cutoffs are appropriate.

In [ ]:
STEP = 20 # bin width / MPa
STRESS_MIN = YOUR RESPONSE # Minimum / MPa
STRESS_MAX = YOUR RESPONSE # Maximum / MPa

BINS = np.arange(STRESS_MIN, STRESS_MAX, STEP)

# names of each group from 1 to 4
group_names = {1: 'Uncoated, Diameter 3-5 mm',
          2: 'Coated, Diameter 3-5 mm',
          3: 'Uncoated, Diameter 7-9 mm',
          4: 'Coated, Diameter 7-9 mm'}


# store a mask of the values that are within the user-defined stress limits
in_range = (df["Flex Strength / MPa"] >= STRESS_MIN) & (df["Flex Strength / MPa"] <= STRESS_MAX)
ignored = ~in_range & (df["group"] != 0)


# print the specimens that are being ignored
print(f"Ignored {ignored.sum()} specimens outside {STRESS_MIN}–{STRESS_MAX} MPa:")
print(df.loc[ignored, ["group", "Flex Strength / MPa"]])
    

#### Option 1: Four subplots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))

for ax, key in zip(axes.ravel(), group_names.keys()):

    # select the data for each group
    group_data = df[df['group'] == key]['Flex Strength / MPa']

    # plot each group's data in a histogram
    counts, bin_edges, patches = ax.hist(group_data, bins=BINS, label=group_names[key], edgecolor='black')
    ax.set_title(group_names[key])
    ax.set_xlabel(YOUR RESPONSE)
    ax.set_ylabel(YOUR RESPONSE)
    ax.set_ylim(YOUR RESPONSE)  # set a consistent y-axis for comparison



plt.tight_layout()
plt.show()

# STUDENT: add a descriptive figure caption. Explain the trends in the caption.
print("Figure 1. DESCRIPTIVE CAPTION HERE")

#### Option 2: A single stacked plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

# collect each group's data into a list
data = [df[df['group'] == key]['Flex Strength / MPa'] for key in group_names.keys()]

# one hist call with all groups -> stacked
ax.hist(data, bins=BINS, stacked=True, label=list(group_names.values()), edgecolor='black')

ax.set_xlabel(YOUR RESPONSE)
ax.set_ylabel(YOUR RESPONSE)

# these two lines align the order of the legend with the bar plot
handles, labels = ax.get_legend_handles_labels() # get the legend information
ax.legend(handles[::-1], labels[::-1]) # flip the order of the handles and labels

plt.tight_layout()
plt.show()

print("Figure 1. DESCRIPTIVE CAPTION HERE")

**`ENGINEERING CHECK`**:

If any specimens were excluded from the plot above, they should be dropped from any further analysis. If you don't want to ignore them, adjust your stress limits and re-run the code above.

Use the cell below to drop specimens ignored in the plot above from future plotting and statistics. 

In [ ]:
# Run this cell to drop all the ignored specimens from future plotting and statistics

df = df[~ignored]

**`YOUR RESPONSE`**

Explain why you selected plotting option 1 or 2. How does it affect your interpretation of the data?

*Replace this text with your response in paragraph form.*

### 3.2 Plot the cumulative probability of failure (20 points)

Plot the cumulative probability that a specimen has failed as a function of flexural strength for the combined data.

To determine the probability of failure, sort the specimens in ascending order of flexural strength, then determine the probability of failure, defined as
$$P\left(  \sigma_\mathrm{fs}  \right) = \frac{n}{N+1}$$
where $n$ is the number of failed samples for a given stress and $N$ is the total number of samples.

**Functions for your code**:

- `np.arange(start, stop)`: create an array from `start` to `stop`, in steps of 1 
- `df.sort_values("column name")`: sort the dataframe by the values in `"column name"` (ascending by default)
- `len(df)`: return the number of rows in the dataframe


In [ ]:
df = df.sort_values("Flex Strength / MPa")

N = YOUR RESPONSE
n = YOUR RESPONSE
probability = YOUR RESPONSE

fig, ax = plt.subplots(figsize=(7, 4))

#STUDENT: add code to plot probability as a function of flexural strength


plt.tight_layout()
plt.show()

print("Figure 2. YOUR RESPONSE")

## 4. Statistics (10 points)

**`YOUR RESPONSE`**

Determine and list the

- number of specimens
- mean
- median
- standard deviation

of the observed flexural strength values for each group and for the lumped dataset.

**Variables for your code**:

- `strengths`: flexural strength for the selected subset of specimens

**Functions for your code**:

- `.count()`: Returns the number of observations / samples
- `.mean()`: Returns the mean
- `.median()`: Returns the median
- `.std()`: Returns the standard deviation

In [ ]:
results = []

for group, title in group_names.items():

    # flexural strengths of each group
    strengths = df.loc[df["group"] == group, "Flex Strength / MPa"]

    # add the statistics for each group to the dataframe
    results.append({
        "Group": title,
        "Count": YOUR RESPONSE,
        "Mean / MPa": YOUR RESPONSE,
        "Median / MPa": YOUR RESPONSE,
        "Std dev / MPa": YOUR RESPONSE
    })


# flexural strengths for the lumped dataset
strengths = df["Flex Strength / MPa"]

# add the statistics for the lumped dataset to the dataframe
results.append({
    "Group": "Entire dataset",
    "Count": YOUR RESPONSE,
    "Mean / MPa": YOUR RESPONSE,
    "Median / MPa": YOUR RESPONSE,
    "Std dev / MPa": YOUR RESPONSE
})

# print the statistics
stats_df = pd.DataFrame(results).round(1)
print(stats_df.to_string(index=False))

## 5. Effect of coating (10 points)

**`YOUR RESPONSE`**

Does coating the specimens with a layer of nail polish have an effect on the stress distribution? If so, explain the physical mechanism resulting in the difference.

*Replace this text with your response in paragraph form.*

## 6. Fracture Surfaces (15 points)

**`YOUR RESPONSE`**

Describe in words and show the most prominent features of the fracture surface. Refer to specific features in your micrograph(s).

*Replace this text with your response in paragraph form.*

In [ ]:
# Include the relevant micrographs here.

# Basic code for displaying an image
micrograph_image = plt.imread("image_file.jpg") # replace image_file.jpg with the correct file

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(micrograph_image)
ax.set_title("YOUR RESPONSE")
ax.axis("off")

plt.tight_layout()
plt.show()

# provide a descriptive caption
print("Figure X. YOUR RESPONSE")

## CONTRIBUTION STATEMENT

**`YOUR RESPONSE`**

*List each contributor and what they did.*

| Contributor | Contribution |
|:---|:---|
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |
| YOUR NAME | YOUR CONTRIBUTION |

## SUBMISSION

When this notebook is complete and ready to submit, export it as a PDF file. If the PDF export fails, check your $\LaTeX$ code for errors.